# iSWAP Module

The iSWAP is a plate transport gripper arm on the Hamilton STAR(let). After {meth}`~pylabrobot.hamilton.liquid_handlers.star.star._HamiltonSTAR.setup`, it is available as `star.iswap` — an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm`.

## Setup

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STARLet

star = STARLet()
await star.setup()

In [ ]:
assert star.iswap is not None, "iSWAP is not available on this robot"

## Deck layout

In [ ]:
from pylabrobot.resources import PLT_CAR_L5AC_A00, Cor_96_wellplate_360ul_Fb

plate_carrier = PLT_CAR_L5AC_A00(name="plate_carrier")
plate_carrier[0] = plate = Cor_96_wellplate_360ul_Fb(name="plate_01")
plate_carrier[1] = Cor_96_wellplate_360ul_Fb(name="plate_02")
star.deck.assign_child_resource(plate_carrier, rails=15)

## Moving resources

### `move_resource`

The simplest way to move a plate between locations. This picks up the resource, moves it, and drops it in one call.

For fine-grained control over pickup and drop, pass `backend_params` with {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.PickUpParams` or {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.DropParams`.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.iswap import iSWAPBackend

await star.iswap.move_resource(plate, plate_carrier[2])

In [ ]:
await star.iswap.move_resource(
  plate, plate_carrier[0],
  backend_params=iSWAPBackend.PickUpParams(grip_strength=8),
)

### `pick_up_resource` and `drop_resource`

For more control, pick up and drop separately. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.PickUpParams` or {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.DropParams` via `backend_params` to control grip strength, traverse height, etc.

In [ ]:
await star.iswap.pick_up_resource(plate)
await star.iswap.drop_resource(plate_carrier[2])

In [ ]:
await star.iswap.pick_up_resource(
  plate,
  backend_params=iSWAPBackend.PickUpParams(grip_strength=8),
)
await star.iswap.drop_resource(
  plate_carrier[0],
  backend_params=iSWAPBackend.DropParams(z_position_at_end=300),
)

### Grip direction

By default, the iSWAP grips from the front. Use the `direction` parameter to change this.

In [ ]:
from pylabrobot.capabilities.arms.standard import GripDirection

await star.iswap.pick_up_resource(plate, direction=GripDirection.LEFT)
await star.iswap.drop_resource(plate_carrier[0], direction=GripDirection.LEFT)

## Common tasks

### Parking

Park the iSWAP. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.ParkParams` via `backend_params` to control the minimum traverse height.

In [ ]:
await star.driver.iswap.park()

In [ ]:
await star.driver.iswap.park(
  backend_params=iSWAPBackend.ParkParams(minimum_traverse_height=350),
)

### Opening the gripper

Open the iSWAP gripper using {meth}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm.open_gripper`. **Warning**: this will release any object that is gripped. Useful for error recovery.

In [ ]:
await star.iswap.open_gripper(gripper_width=130)

### Closing the gripper

Close the iSWAP gripper. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.CloseGripperParams` via `backend_params` to control grip strength and plate width tolerance.

In [ ]:
await star.iswap.close_gripper(gripper_width=85)

In [ ]:
await star.iswap.close_gripper(
  gripper_width=85,
  backend_params=iSWAPBackend.CloseGripperParams(grip_strength=8, plate_width_tolerance=1.0),
)

## Rotations

You can rotate the iSWAP to 12 predefined positions using {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.rotate`.

![iSWAP positions](img/iswap_positions.png)

The `rotate` method takes a `rotation_drive` orientation and the final `grip_direction` of the iSWAP (with respect to the deck). The internal motion planner automatically adjusts the wrist drive.

In [ ]:
from pylabrobot.capabilities.arms.standard import GripDirection

await star.driver.iswap.rotate(
  rotation_drive=iSWAPBackend.RotationDriveOrientation.LEFT,
  grip_direction=GripDirection.RIGHT,
)

### Controlling the wrist and rotation drive individually

You can also control the wrist (T-drive) and rotation drive (W-drive) individually.

In [ ]:
# Rotation drive: LEFT, RIGHT, or FRONT
# Wrist drive: LEFT, RIGHT, STRAIGHT, or REVERSE
await star.driver.iswap.rotate_rotation_drive(iSWAPBackend.RotationDriveOrientation.LEFT)
await star.driver.iswap.rotate_wrist(iSWAPBackend.WristDriveOrientation.REVERSE)

## Slow movement

Use the {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.slow` context manager to reduce speed during sensitive operations. Pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.MoveToLocationParams` via `backend_params` for additional control over acceleration and collision avoidance.

In [ ]:
async with star.driver.iswap.slow():
  await star.iswap.move_resource(plate, plate_carrier[2])

In [ ]:
async with star.driver.iswap.slow():
  await star.iswap.move_resource(
    plate, plate_carrier[0],
    backend_params=iSWAPBackend.MoveToLocationParams(collision_control_level=1),
  )

## Manual movement (teaching / calibration)

### 1. Clear the Y range

For safety, move the other components as far away as possible before teaching.

In [ ]:
await star.driver.pip.position_components_for_free_iswap_y_range()

### 2. Set rotation

Move the iSWAP wrist and rotation drive to the correct orientation as [explained above](#rotations). Be careful to move the iSWAP to a position where it does not hit any other components.

### 3. Absolute movement

Use the following methods to move the iSWAP to absolute X, Y and Z positions. All units are in mm.

In [ ]:
await star.driver.iswap.move_x(x=200)

In [ ]:
await star.driver.iswap.move_y(y=200)

In [ ]:
await star.driver.iswap.move_z(z=200)

### 4. Computing plate position from calibration

The x, y, and z values refer to the **center** of the iSWAP gripper, making them agnostic to plate size. In PLR, however, all locations are with respect to the left-front-bottom (LFB) of the plate. To convert from the calibrated center to LFB, subtract the plate's center-bottom anchor:

In [ ]:
from pylabrobot.resources import Coordinate

x, y, z = 200, 200, 200  # calibrated center position
calibrated_position = Coordinate(x, y, z)
plate_lfb_absolute = calibrated_position - plate.get_anchor("c", "c", "b")

The plate's LFB is now in absolute coordinates. If the plate is a child of some parent resource, compute the relative location by subtracting the parent's absolute position:

In [ ]:
parent = plate_carrier  # example: the plate's parent resource
parent_absolute = parent.get_location_wrt(star.deck)
plate_relative = plate_lfb_absolute - parent_absolute

Use this with `parent.assign_child_resource(plate, location=plate_relative)` to place the plate at the calibrated position.

### 5. Relative movements

You can also move the iSWAP relative to its current position. All units are in mm. These refer to the center of the gripper.

In [ ]:
await star.driver.iswap.move_relative_x(step_size=10)

In [ ]:
await star.driver.iswap.move_relative_y(step_size=10)

In [ ]:
await star.driver.iswap.move_relative_z(step_size=10)

## Teardown

In [ ]:
await star.stop()